# Mutations Preprocessing — METABRIC
Input: `preprocessed/metabric_mutations_cleaned.csv` (binary 0/1)

Output: single composite dataset → `03_METABRIC_external_validation/mutations/statistical_filtered/`

In [2]:
from pathlib import Path

_cwd = Path().resolve()
BASE_DIR = next(
    p for p in [_cwd, *_cwd.parents]
    if (p / "preprocessed" / "metabric_mutations_cleaned.csv").exists()
)

MUT_FILE     = BASE_DIR / "preprocessed" / "metabric_mutations_cleaned.csv"
OUTCOME_FILE = BASE_DIR / "preprocessed" / "outcome.csv"
OUTPUT_DIR   = BASE_DIR / "mutations" / "statistical_filtered"
STATS_CACHE  = BASE_DIR / "mutations" / "mut_statistics_cache.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_BROAD = 1000
MIN_GENES    = 10

assert MUT_FILE.exists(),     f"Not found: {MUT_FILE}"
assert OUTCOME_FILE.exists(), f"Not found: {OUTCOME_FILE}"

print(f"Base   : {BASE_DIR}")
print(f"Output : {OUTPUT_DIR}")
print("Paths OK")

Base   : C:\Users\olegk\Desktop\Thesis_v3\03_METABRIC_external_validation
Output : C:\Users\olegk\Desktop\Thesis_v3\03_METABRIC_external_validation\mutations\statistical_filtered
Paths OK


In [3]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, rankdata
from statsmodels.stats.multitest import multipletests
from sklearn.feature_selection import mutual_info_regression
import warnings
warnings.filterwarnings("ignore")
print("Imports OK")

Imports OK


## 1. Load data

In [5]:
print("Loading mutations...")
mut = pd.read_csv(MUT_FILE, index_col=0)
print(f"  Shape  : {mut.shape}  (samples x genes, binary 0/1)")
print(f"  Values : {sorted(mut.values.flatten()[:100].astype(int).tolist()[:5])} (should be 0/1)")

print("\nLoading outcome...")
outcome = pd.read_csv(OUTCOME_FILE, index_col=0)
print(f"  Samples : {len(outcome)}")
print(f"  Events  : {int(outcome['OS'].sum())} ({outcome['OS'].mean()*100:.1f}%)")

common  = sorted(set(mut.index) & set(outcome.index))
mut     = mut.loc[common].copy()
outcome = outcome.loc[common].copy()
print(f"  Overlap : {len(common)} samples")

Loading mutations...
  Shape  : (1980, 116)  (samples x genes, binary 0/1)
  Values : [0, 0, 0, 0, 0] (should be 0/1)

Loading outcome...
  Samples : 1980
  Events  : 1143 (57.7%)
  Overlap : 1980 samples


## 2. Mutation frequency analysis

In [7]:
freq_pct = (mut.sum(axis=0) / len(mut) * 100).sort_values(ascending=False)

print(f"Total genes: {len(freq_pct)}")
print(f"\nFrequency thresholds:")
for t in [10, 5, 2.5, 2, 1, 0.5]:
    print(f"  >= {t:4.1f}% : {(freq_pct >= t).sum():4d} genes")

print(f"\nTop 20 most mutated genes:")
for gene, pct in freq_pct.head(20).items():
    print(f"  {gene:<20} {pct:5.1f}%  (n={int(mut[gene].sum())})")

# Remove zero-variance (never or always mutated)
var = mut.var(axis=0)
mut = mut.loc[:, var > 0]
freq_pct = freq_pct[freq_pct.index.isin(mut.columns)]
print(f"\nAfter removing zero-variance: {mut.shape[1]} genes")

Total genes: 116

Frequency thresholds:
  >= 10.0% :    8 genes
  >=  5.0% :   23 genes
  >=  2.5% :   55 genes
  >=  2.0% :   69 genes
  >=  1.0% :  116 genes
  >=  0.5% :  116 genes

Top 20 most mutated genes:
  PIK3CA                40.2%  (n=795)
  TP53                  33.2%  (n=658)
  MUC16                 16.5%  (n=326)
  AHNAK2                15.7%  (n=310)
  KMT2C                 11.7%  (n=231)
  SYNE1                 11.7%  (n=231)
  GATA3                 11.5%  (n=228)
  MAP3K1                10.0%  (n=198)
  DNAH11                 8.8%  (n=175)
  AHNAK                  8.8%  (n=175)
  CDH1                   8.5%  (n=168)
  DNAH2                  7.4%  (n=147)
  MLL2                   7.1%  (n=140)
  USH2A                  7.0%  (n=139)
  RYR2                   6.8%  (n=135)
  DNAH5                  6.6%  (n=130)
  HERC2                  6.3%  (n=125)
  PDE4DIP                5.9%  (n=117)
  AKAP9                  5.9%  (n=117)
  TG                     5.7%  (n=112)

After r

## 3. Statistical scoring — Spearman + MI + Distance correlation

In [9]:
if STATS_CACHE.exists():
    print(f"Loading cached stats from {STATS_CACHE.name}...")
    stats = pd.read_csv(STATS_CACHE)
    stats = stats[stats["gene"].isin(mut.columns)].reset_index(drop=True)
    print(f"  Loaded {len(stats):,} genes")

else:
    genes   = mut.columns.tolist()
    os_time = outcome["OS.time"].values
    n       = len(genes)
    print(f"Computing stats for {n:,} genes...")

    # Spearman + FDR
    print("  [1/3] Spearman...")
    rho_arr  = np.zeros(n)
    pval_arr = np.zeros(n)
    for i, g in enumerate(genes):
        r, p        = spearmanr(mut[g].values, os_time)
        rho_arr[i]  = r if not np.isnan(r) else 0.0
        pval_arr[i] = p if not np.isnan(p) else 1.0
    _, fdr_arr, _, _ = multipletests(pval_arr, method="fdr_bh")
    print(f"    FDR < 0.05: {(fdr_arr < 0.05).sum()} genes")

    # Mutual information
    print("  [2/3] Mutual information...")
    mi_arr = mutual_info_regression(mut.values, os_time, random_state=42, n_neighbors=5)

    # Distance correlation (rank approximation)
    print("  [3/3] Distance correlation...")
    os_rank = rankdata(os_time).astype(float)
    dc_arr  = [
        abs(float(np.corrcoef(rankdata(mut[g].values).astype(float), os_rank)[0, 1]))
        for g in genes
    ]

    stats = pd.DataFrame({
        "gene":          genes,
        "spearman_rho":  rho_arr,
        "abs_spearman":  np.abs(rho_arr),
        "pval":          pval_arr,
        "fdr":           fdr_arr,
        "mutual_info":   mi_arr,
        "distance_corr": dc_arr,
        "freq_pct":      freq_pct.reindex(genes).values,
    })

    stats["rank_spearman"] = rankdata(-stats["abs_spearman"])
    stats["rank_mi"]       = rankdata(-stats["mutual_info"])
    stats["rank_dcor"]     = rankdata(-stats["distance_corr"])
    stats["composite"]     = (stats["rank_spearman"] + stats["rank_mi"] + stats["rank_dcor"]) / 3.0
    stats = stats.sort_values("composite").reset_index(drop=True)

    STATS_CACHE.parent.mkdir(parents=True, exist_ok=True)
    stats.to_csv(STATS_CACHE, index=False)
    print(f"  Cached to {STATS_CACHE}")

print(f"\nTop 10 by composite score:")
print(stats[["gene", "freq_pct", "abs_spearman", "mutual_info", "composite"]].head(10).to_string(index=False))

Computing stats for 116 genes...
  [1/3] Spearman...
    FDR < 0.05: 3 genes
  [2/3] Mutual information...
  [3/3] Distance correlation...
  Cached to C:\Users\olegk\Desktop\Thesis_v3\03_METABRIC_external_validation\mutations\mut_statistics_cache.csv

Top 10 by composite score:
  gene  freq_pct  abs_spearman  mutual_info  composite
  TP53 33.232323      0.134493     0.024452   1.000000
 GATA3 11.515152      0.100650     0.006080   2.666667
    TG  5.656566      0.069029     0.004591   5.666667
  CBFB  4.646465      0.098938     0.003418   7.333333
 NCOR2  4.393939      0.054497     0.004316   8.666667
 LAMA2  4.696970      0.053886     0.001864  15.333333
  SIK2  1.060606      0.067165     0.000752  17.666667
THSD7A  2.626263      0.031150     0.004134  19.666667
  TAF1  2.020202      0.050797     0.000616  22.000000
 AHNAK  8.838384      0.031408     0.002138  22.333333


## 4. Build V8 composite dataset

In [11]:
# Top TARGET_BROAD genes by composite score
top_genes = stats.head(TARGET_BROAD)["gene"].tolist()
top_genes = [g for g in top_genes if g in mut.columns]

# Pad if fewer than MIN_GENES
if len(top_genes) < MIN_GENES:
    have = set(top_genes)
    top_genes += [g for g in stats["gene"] if g not in have][:MIN_GENES - len(top_genes)]
    print(f"Padded to {len(top_genes)} genes")

df_v8 = mut[top_genes]

fname = f"mut_8_composite_{len(df_v8.columns)}genes.csv"
df_v8.to_csv(OUTPUT_DIR / fname)
outcome.to_csv(OUTPUT_DIR / "outcome.csv")
stats.to_csv(OUTPUT_DIR / "mut_statistics_complete.csv", index=False)

print(f"Genes    : {len(df_v8.columns)}")
print(f"Samples  : {len(df_v8)}")
print(f"Saved    : {fname}")
print(f"\nFrequency distribution of selected genes:")
sel_freq = freq_pct[df_v8.columns].describe()
print(sel_freq.to_string())

Genes    : 116
Samples  : 1980
Saved    : mut_8_composite_116genes.csv

Frequency distribution of selected genes:
count    116.000000
mean       3.958116
std        5.263095
min        1.010101
25%        1.502525
50%        2.323232
75%        4.204545
max       40.151515


## 5. Verification

In [13]:
df_check = pd.read_csv(OUTPUT_DIR / fname, index_col=0)

assert list(df_check.index) == list(outcome.index), "Index mismatch!"
assert df_check.isna().sum().sum() == 0, "NaN values found!"
assert set(df_check.values.flatten().astype(int)).issubset({0, 1}), "Non-binary values!"

print(f"Shape   : {df_check.shape}")
print(f"Binary  : OK (values are 0/1)")
print(f"No NaN  : OK")
print(f"Index   : OK")
print(f"\nTop 10 genes by composite score:")
print(stats[["gene", "freq_pct", "abs_spearman", "mutual_info", "composite"]].head(10).to_string(index=False))
print(f"\nNext step: run 03_run_MB_mutations_METABRIC.py")

Shape   : (1980, 116)
Binary  : OK (values are 0/1)
No NaN  : OK
Index   : OK

Top 10 genes by composite score:
  gene  freq_pct  abs_spearman  mutual_info  composite
  TP53 33.232323      0.134493     0.024452   1.000000
 GATA3 11.515152      0.100650     0.006080   2.666667
    TG  5.656566      0.069029     0.004591   5.666667
  CBFB  4.646465      0.098938     0.003418   7.333333
 NCOR2  4.393939      0.054497     0.004316   8.666667
 LAMA2  4.696970      0.053886     0.001864  15.333333
  SIK2  1.060606      0.067165     0.000752  17.666667
THSD7A  2.626263      0.031150     0.004134  19.666667
  TAF1  2.020202      0.050797     0.000616  22.000000
 AHNAK  8.838384      0.031408     0.002138  22.333333

Next step: run 03_run_MB_mutations_METABRIC.py
